## Load The Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

columns = [
    "EmployeeId","FirstName", "LastName", "HireDate", "BirthDate",
     "Sales", "Revenue", "Avg Revenue per Invoice"
]

# Read the CSV and assign column names
df = pd.read_csv("newdb.csv", header=None, names=columns, sep=";")

df

## Convert date variables to datetime format

In [ ]:
df['HireDate'] = pd.to_datetime(df['HireDate'])
df['BirthDate'] = pd.to_datetime(df['BirthDate'])

df.head()

## Drop name from table

In [ ]:
df = df.drop(columns=['FirstName', 'LastName'])
df

## Create variables age and tenure

In [ ]:
df['Tenure'] = ((pd.Timestamp.now() - df['HireDate']).dt.days)//30
df["Age"] = ((pd.Timestamp.now() - df['BirthDate']).dt.days)//365
df = df.drop(columns=["HireDate", "BirthDate"])
df

Creating performance labels

In [ ]:
df['PerformanceLabel'] = pd.qcut(df['Revenue'], q=3, labels=[0, 1, 2])

performance_summary = df.groupby('PerformanceLabel')[['Revenue', 'Sales', 'Avg Revenue per Invoice']].mean()
print("Performance Summary (Revenue-based labels):")
print(performance_summary)

## Dropping not needed columns


In [ ]:
df = df.drop(columns=['Revenue', 'EmployeeId'])
df

## Density plots


In [ ]:
numeric_columns = df.select_dtypes(include=['float64', 'int64']).columns

# Create subplots
num_vars = len(numeric_columns)
rows = (num_vars + 1) // 2  # Calculate the number of rows (2 plots per row)
fig, axes = plt.subplots(rows, 2, figsize=(12, 6 * rows))  # Adjust figure size
axes = axes.flatten()  # Flatten axes to make it easier to iterate

# Plot each variable
for i, column in enumerate(numeric_columns):
    sns.kdeplot(df[column], shade=True, ax=axes[i])
    axes[i].set_title(f'Density Plot for {column}')
    axes[i].set_xlabel(column)
    axes[i].set_ylabel('Density')

# Remove empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Adjust layout
plt.tight_layout()
plt.show()

## Splitting into inputs and outputs

In [ ]:
y = df['PerformanceLabel']
x = df.drop(columns=['PerformanceLabel'])
print(x.head())

## Standardisation

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x = scaler.fit_transform(x)

x = pd.DataFrame(x, columns=['Sales', 'Avg Revenue per Invoice', 'Tenure', 'Age'])
x

## Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Check the shapes to ensure correct splitting
print("X_train shape:", x_train.shape)
print("X_test shape:", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## Training the models

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(x_train, y_train)
dt_predictions = dt_model.predict(x_test)
dt_accuracy = accuracy_score(y_test, dt_predictions)

# Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(x_train, y_train)
rf_predictions = rf_model.predict(x_test)
rf_accuracy = accuracy_score(y_test, rf_predictions)

# Support Vector Machine
svm_model = SVC(random_state=42)
svm_model.fit(x_train, y_train)
svm_predictions = svm_model.predict(x_test)
svm_accuracy = accuracy_score(y_test, svm_predictions)

# Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)  # Increased max_iter for convergence
lr_model.fit(x_train, y_train)
lr_predictions = lr_model.predict(x_test)
lr_accuracy = accuracy_score(y_test, lr_predictions)

# Naive Bayes
nb_model = GaussianNB()
nb_model.fit(x_train, y_train)
nb_predictions = nb_model.predict(x_test)
nb_accuracy = accuracy_score(y_test, nb_predictions)

#XGBoost

xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(x_train, y_train)
xgb_predictions = xgb_model.predict(x_test)
xgb_accuracy = accuracy_score(y_test, xgb_predictions)




# Print the accuracies
print("Decision Tree Accuracy:", dt_accuracy)
print("Random Forest Accuracy:", rf_accuracy)
print("SVM Accuracy:", svm_accuracy)
print("Logistic Regression Accuracy:", lr_accuracy)
print("Naive Bayes Accuracy:", nb_accuracy)
print("XGBoost Accuracy:", xgb_accuracy)


## Cross Validation on RF, LR and XGBoost

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# List of models to perform 5-fold cross-validation on
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=10000),  # Increased max_iter for convergence
    'XGBoost': XGBClassifier(random_state=42)
}

# Perform 5-fold cross-validation for each model
for model_name, model in models.items():
    # Get the cross-validation scores (accuracy scores for each fold)
    cv_scores = cross_val_score(model, x_train, y_train, cv=5, scoring='accuracy')
    # Calculate the average accuracy across all folds
    mean_cv_accuracy = cv_scores.mean()
    # Print the average cross-validation accuracy
    print(f"{model_name} 5-Fold Cross-Validation Accuracy: {mean_cv_accuracy:.4f}")

## Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


# Define parameter grids for each model
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'random_state': [42]
}

lr_param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'saga'],
    'max_iter': [1000]
}

xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 0.9, 1.0],
    'random_state': [42]
}

# Create models
rf_model = RandomForestClassifier(random_state=42)
lr_model = LogisticRegression(random_state=42, max_iter=1000)
xgb_model = XGBClassifier(random_state=42)

# Perform GridSearchCV for each model
# For Random Forest
rf_grid_search = GridSearchCV(rf_model, rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
rf_grid_search.fit(x_train, y_train)
print("Best Random Forest Hyperparameters:", rf_grid_search.best_params_)
print("Best Random Forest Accuracy:", rf_grid_search.best_score_)

# For Logistic Regression
lr_grid_search = GridSearchCV(lr_model, lr_param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
lr_grid_search.fit(x_train, y_train)
print("Best Logistic Regression Hyperparameters:", lr_grid_search.best_params_)
print("Best Logistic Regression Accuracy:", lr_grid_search.best_score_)

# For XGBoost
xgb_grid_search = GridSearchCV(xgb_model, xgb_param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
xgb_grid_search.fit(x_train, y_train)
print("Best XGBoost Hyperparameters:", xgb_grid_search.best_params_)
print("Best XGBoost Accuracy:", xgb_grid_search.best_score_)

# After hyperparameter tuning, you can use the best models to make predictions on the test set
best_rf_model = rf_grid_search.best_estimator_
best_lr_model = lr_grid_search.best_estimator_
best_xgb_model = xgb_grid_search.best_estimator_


## Fit the final XGBoost model

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# Assume the best XGBoost model is stored in best_xgb_model
# Fit the model on the training data
best_xgb_model.fit(x_train, y_train)

# Make predictions on the test data
xgb_predictions = best_xgb_model.predict(x_test)

# Evaluate the model using accuracy, precision, recall, and f1 score
accuracy = accuracy_score(y_test, xgb_predictions)
precision = precision_score(y_test, xgb_predictions, average='weighted')
recall = recall_score(y_test, xgb_predictions, average='weighted')
f1 = f1_score(y_test, xgb_predictions, average='weighted')

# Print the evaluation metrics
print("XGBoost Model Performance:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Performance metrics for plotting
metrics = [accuracy, precision, recall, f1]
metrics_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

# Plot the bar chart for model performance metrics
plt.figure(figsize=(8, 6))
plt.bar(metrics_labels, metrics, color='#12726D')
plt.title('XGBoost Model Performance Metrics')
plt.xlabel('Metrics')
plt.ylabel('Score')
plt.ylim([0.6, 1])  # Set y-axis limit from 0.8 to 1
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_test, xgb_predictions)
custom_cmap = LinearSegmentedColormap.from_list("custom_green", ["#ffffff", "#12726D"])

# Plot the confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap=custom_cmap, xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
plt.title("Confusion Matrix - XGBoost")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

## Feature Importance

In [ ]:
# Get feature importance from the model
feature_importance = best_xgb_model.get_booster().get_score(importance_type='weight')

# Convert to a sorted list
sorted_importance = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)

# Extract feature names and their importance
features = [item[0] for item in sorted_importance]
importance_scores = [item[1] for item in sorted_importance]

# Plot the feature importance
plt.figure(figsize=(10, 6))
plt.barh(features, importance_scores, color='#12726D')
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("XGBoost Feature Importance")
plt.gca().invert_yaxis()  # Invert y-axis for better readability
plt.show()